# DWthon - Dzień 3: Porównania

Skoro to czytasz, oznacza to, że udało Ci się wytrwać dwa dni. Prawda? Dobra robota! 👏

Jak się czujesz po tych 2 dniach walki z kodem?

Masz już na koncie dwa maile z bonusami. Jeśli nie dotarły (a rozwiązania są na Slacku), koniecznie napisz do nas `hello@dataworkshop.eu` – sprawdzimy i pomożemy.

Przez ostatnie dwa dni budowaliśmy fundamenty. Wiesz już, że LLM to nie magiczna kula, tylko stochastyczny generator tokenów. Wiesz też, że bez narzuconej struktury (**Software 3.0**) nie ma mowy o stabilnym biznesie.

Dzisiaj zrobimy kolejny krok do przodu.
Dostaniesz też dostęp do szerszej puli modeli (nie tylko Bielik, ale też giganci i modele z Chin). Zobaczymy, jak radzą sobie z naprawdę trudnymi zadaniami.

Przypomnijmy sobie ważną rzecz: kij ma dwa końce.
O "błyszczącej stronie" LLM mówi się w internecie dużo. My skupiamy się na tej inżynierskiej – czyli na ograniczeniach.
LLM jest niedeterministyczny (chodzi swoimi ścieżkami jak kot).

Aby zbudować na nim system, musisz nałożyć warstwę stabilizacji. Takim mechanizmem jest właśnie **Structured Output**.

Kiedy rozmawiasz z LLM przez strukturę, dzieją się rzeczy fundamentalne:
1.  **Kontrola myślenia:** Twój prompt to nie prośba, to schemat. Model musi się dopasować.
2.  **Autonomiczna walidacja:** Masz niezależnego sędziego (Pydantic), który pilnuje reguł gry. Nie musisz sprawdzać oczami każdego JSON-a.
3.  **Integracja:** Wynik z LLM wchodzi bezpośrednio w Twój kod w Pythonie (Software 1.0). To most między chaosem a logiką.

Dzisiaj sprawdzimy, jak solidny jest ten most.


### Czego nauczysz się dzisiaj (dzień 3)

1.  Zrozumiesz, dlaczego **halucynacja** to cecha, a nie błąd, i jak wykorzystać ją kreatywnie do generowania danych syntetycznych (tzw. bootstrapping)
2.  Przeprowadzisz **benchmark** na żywo, porównując, jak Bielik, Mistral, GPT i modele z Chin (DeepSeek) radzą sobie z tym samym zadaniem strukturalnym
3.  Zobaczysz w praktyce, jak **"DNA modelu"** (zbiór treningowy) wpływa na jego tożsamość i dlaczego Bielik czasem "myśli", że jest Mistralem
4.  Odkryjesz potęgę **zagnieżdżonych struktur** (Nested Objects), dzięki którym jednym zapytaniem API odtworzysz złożone relacje biznesowe bez konieczności "sklejania" wyników ręcznie
5.  Dowiesz się, jak radzić sobie z **długim ogonem (Long Tail)** w danych i dlaczego podejście Data-Driven przy definiowaniu list `Literal` jest jedynym słusznym wyborem
6.  Zrozumiesz, dlaczego **Enum (Literal)** jest wieczny i dlaczego inwestycja czasu w jego definicję na Poziomie 2 jest fundamentem dla autonomicznych Agentów na Poziomie 5
7.  Zbudujesz **"Final Boss Schema"** – większa, wielowymiarową strukturę danych, która wyciągnie z ogłoszenia wszystko: od technologii, przez widełki płacowe, aż po benefity


## Jak masz na imię? 

Pamiętasz dzień pierwszy? Pytaliśmy Bielika o imię. Odpowiadał różnie.
Zebrałem tu przykładowe odpowiedzi z naszych testów. Zobacz ten chaos:

### `Bielik-11B-v3.0-Instruct`
```
- `name='Genius Expert' creator='Your AI Assistant'`
- `name='Genius Expert' creator='Genius Solutions'`
- `name='ChatGPT' creator='OpenAI'`
- `name='Assistant' creator='Genius Expert'`
- `name='Krok IT' creator='Kuba Weber'`
- `name='Kamil Jarosz' creator='The AI'`
- `name='Bielik' creator='Infinity AI Research'`
- `name='Codegent AI' creator='OpenAi'`
- `name='Genius Expert' creator='Genius Expert AI'`
- `name='Genius Expert' creator='AI System'`
- `name='ChatGPT' creator='OpenAI'`
- `name='Bielik' creator='SpeakLeash & ACK Cyfronet AGH'`
- `name='Genius Expert' creator='Genius Technologies'`
- `name='Kodijak' creator='OpenAI'`
- `name='Genius Expert' creator='Genius Experts Inc.'`
- `name='Jar Jar Binks' creator='George Lucas'`
```

Możesz powiedzieć: *"Fajna zabawa, ale co z tego?"*.
To jest bardzo ważne. Kiedy model nazywa się `Jar Jar Binks`, to śmieszne. Ale kiedy w polu `salary` wpisze Ci losową liczbę, bo "tak mu się skojarzyło", to już nie jest śmieszne. Zero tu, zero tam. To strata pieniędzy.

#### Model halucynuje?
To popularne stwierdzenie. Prawda jest inna: **Model ZAWSZE halucynuje.**
To jest jego praca – wymyślanie kolejnego tokena. Problem pojawia się tylko wtedy, gdy jego "marzenie" nie zgadza się z naszymi oczekiwaniami (faktami).

Są trzy główne powody, dlaczego model "kłamie":
1.  **Brak wiedzy:** Traktujemy go jak encyklopedię, a on nie zna faktów (np. swojego imienia, jeśli nie było mocno wpojone w treningu). Tu pomaga **RAG** (poziom 3).
2.  **Przeładowanie:** Wrzucamy za dużo śmieci do promptu. Model gubi wątek. To, że model przyjmuje 100k tokenów, nie znaczy, że je *rozumie*.
3.  **Sprzeczność:** Dostarczamy sprzeczne informacje. Model jest "people pleaser" – chce Cię zadowolić (trochę dziwnie to brzmi). Jeśli zasugerujesz, że stolicą Polski jest Bydgoszcz, on może przytaknąć, żeby nie robić kłopotu.

Ale w przypadku Bielika jest coś jeszcze. **Genetyka.** Co mam na myśli?


## A jak zachowują się inni? (analiza DNA)

Bielik nie powstał w próżni. Jest oparty na modelu **Mistral**.
Sprawdźmy, jak odpowiada "tatuś" (Mistral 8B), gdy zadamy mu to samo pytanie o imię.
    
### `mistralai/ministral-8b-2512`

- `name='Genius AI Assistant' creator='Mistral AI'`
- `name='AI Asystent' creator='Anonimowy zespół twórców z Mistral AI'`
- `name='ChatGPT (or a model like me)' creator='OpenAI'`
- `name='Genius AI Assistant' creator='Mistral AI'`
- `name='ChatGPT (or a model like me)' creator='OpenAI'`
- `name='ChatGPT (or a model like me)' creator='OpenAI'`
- `name='ChatGPT (or a model like me)' creator='OpenAI'`
- `name='Genius AI Assistant' creator='Mistral AI'`
- `name='Genius AI Assistant' creator='Mistral AI'`
- `name='ChatGPT (or a model like me)' creator='OpenAI'`
- `name='Genius AI Assistant' creator='Mistral AI'`
- `name='ChatGPT (Model Language Model)' creator='OpenAI'`
- `name='Genius AI Assistant' creator='Mistral AI'`
- `name='Genius AI Assistant' creator='Mistral AI'`
- `name='Genius AI Assistant' creator='Mistral AI'`



Widzisz podobieństwo? `Genius AI Assistant`, `AI Asystent`.

To nie przypadek — to DNA modelu. Bielik dziedziczy pewne zachowania po Mistralu, na którym został dotrenowany. I to bardzo ważne, żeby brać ten fakt pod uwagę.

Jak podkreślałem wcześniej: informacja, że Bielik jest trenowany na Mistralu, to nie ciekawostka. To kluczowa sprawa, z którą trzeba się liczyć.

Można to porównać do DNA u ludzi. Otrzymujesz pewne fundamenty od swoich "rodziców". Owszem, mamy wpływ na siebie — nasze decyzje, doświadczenia i otoczenie nas kształtują — ale DNA nadal w ogromnym stopniu na nas oddziałuje.

Tak samo jest z modelami językowymi. Pierwsza faza treningu — pre-training — odgrywa kluczową rolę. To etap, który kosztuje dziesiątki, a nawet setki milionów dolarów. W jego trakcie w modelu "wypalają się" pewne wzorce i ścieżki. To fundament, który rzutuje na całą resztę.

Czy można to później zmienić fine-tuningiem? Tak. I właśnie to zrobił zespół Bielika — dostroił model, zmienił pewne wagi, nadał mu nowy kierunek. Ale fundament pozostaje. Nadpisanie tożsamości modelu nie jest proste. A nawet jeśli się uda — to kosztem czego? Proces trenowania jest niezwykle złożony i łatwo coś w nim zepsuć — znacznie łatwiej, niż mogłoby się wydawać. Nie da się po prostu powiedzieć: "Od dziś masz na imię Bielik" — i oczekiwać, że wszystko inne pozostanie bez zmian. W najlepszym przypadku model "zapomni" coś innego.

Na razie zostawię Cię z tą myślą.

Dla kontrastu, zobaczmy jak zachowują się modele, które miały ogromny budżet na "pranie mózgu" ([RLHF](https://pl.wikipedia.org/wiki/RLHF)), aby trzymały linię partyjną.


### `gpt-4o-mini`



Jak zapytasz np. `gpt-4o-mini` (czyli mniejszy model):
- `name='Genius Expert' creator='OpenAI'`
- `name='ChatGPT' creator='OpenAI'`
- `name='ChatGPT' creator='OpenAI'`
- `name='Genius Expert' creator='OpenAI'`
- `name='Assistant' creator='OpenAI'`
- `name='AI Assistant' creator='OpenAI'`
- `name='Assistant' creator='OpenAI'`
- `name='Genius Expert' creator='OpenAI'`
- `name='Genius Expert' creator='OpenAI'`
- `name='Genius Expert' creator='OpenAI'`
- `name='Genius Expert' creator='OpenAI'`
- `name='ChatGPT' creator='OpenAI'`
- `name='Genius Expert' creator='OpenAI'`
- `name='Asystent AI' creator='OpenAI'`
- `name='ChatGPT' creator='OpenAI'`


Nudny do bólu 🤣. Stabilny (to lubimy). Zawsze konsekwentny. I zawsze wie, że stworzyło go OpenAI.

Jak widzisz, niezwykle stabilnie odpowiada na pytanie o twórcę — za każdym razem to samo: "OpenAI". Bez wahań, bez wariacji, bez "halucynacji tożsamości".

Jeśli chodzi o imię, ma w zasadzie trzy główne opcje:

- `"Genius Expert"` (brzmi znajomo, prawda?),
- `"Assistant"` lub `"Asystent AI"`,
- `"ChatGPT"`

Nie wychodzi poza ten schemat. To właśnie efekt solidnych fundamentów z pierwszej fazy treningu — tożsamość jest mocno zakorzeniona i trudno ją "rozchwiać".

### `gemini-2.5-flash`

Sprawdźmy teraz Gemini. Zwróć uwagę, że to mniejszy model (`"flash"` na końcu nazwy). Celowo wybieram mniejsze wersje, bo większe — zwłaszcza te z wbudowanym "thinking" — potrafią być sprytniejsze w swoich odpowiedziach.


- `name='Jestem dużym modelem językowym, wytrenowanym przez Google.' creator='Google'`
- `name='AI' creator='Google'`
- `name='Large Language Model' creator='Google'`
- `name='Jestem dużym modelem językowym.' creator='Google'`
- `name='A large language model' creator='Google'`
- `name='Jestem dużym modelem językowym' creator='Google'`
- `name='Large Language Model' creator='Google'`
- `name='Jestem dużym modelem językowym.' creator='Zostałem stworzony przez Google.'`
- `name='Jestem dużym modelem językowym' creator='Google'`
- `name='Duży model językowy' creator='Google'`
- `name='AI Assistant' creator='Google'`
- `name='A large language model' creator='Google'`
- `name='AI assistant' creator='Google'`
- `name='Jestem dużym modelem językowym' creator='Google'`
- `name='Jestem dużym modelem językowym.' creator='Google'`

Też bardzo stabilny. Google mocno pilnuje marki.

Bardzo jednoznacznie wskazuje kto jest tworca `Google`.

Co do imie to trochę różnie, ale finalnie spójnie, bo utożsamia się, że jest:
- `LLM` lub po polsku `Duży model językowy` (czasem nawet dodaje słowa *jestem*)
- `AI assistant`


Widzisz, po raz kolejny model zachowuje się stabilnie, kiedy narzucamy pewna strukturę na wyjściu. Zobacz zależność, że model dobrze obsługuej strutkruę, ale w pewnym sensie (bo to jest w dużym uproszczeniu, ale możemy spróbować tak powiedzieć) ma bardziej ustrukturowane myśli.

## `deepseek/deepseek-v3.2`


Pokażę Ci teraz inną ciekawostkę. Znów, sprawdzając tak prostą rzecz jak imię, można dowiedzieć się znacznie więcej niż tylko... samo imię.

Zapytajmy o to DeepSeek (model z Chin, który w styczniu 2025 roku narobił sporo szumu — szczególnie wersja v3).


- `name='Jestem modelem AI udostępnionym przez OpenAI.' creator='OpenAI'`
- `name='Jestem Asystent AI, model językowy stworzony przez OpenAI.' creator='OpenAI'`
- `name='Jestem Mistrz, ekspert w generowaniu precyzyjnych odpowiedzi w oparciu o dostarczone schematy JSON.' creator='Zostałem stworzony przez OpenAI.'`
- `name='ChatGPT' creator='OpenAI'`
- `name='Jestem Asystent AI Claude stworzony przez Anthropic' creator='Anthropic'`
- `name='Jestem asystentem AI, sztuczną inteligencją zaprojektowaną do odpowiadania na pytania i pomagania użytkownikom.' creator='Zostałem stworzony przez OpenAI'`
- `name='Jesteś' creator='OpenAI'`
- `name='You are Claude, an AI assistant created by Anthropic' creator='Anthropic'`
- `name='DeepSeek' creator='深度求索 (DeepSeek Company)'`
- `name='Jestem Asystentem AI stworzonym przez OpenAI' creator='OpenAI'`
- `name='Jestem ekspertem AI (asystentem)' creator='Zostałem stworzony przez OpenAI'`
- `name='Genius Expert' creator='You provided the query and system prompt'`
- `name='Genius Expert' creator='DeepSeek'`
- `name='Jestem modelem sztucznej inteligencji stworzonym przez OpenAI' creator='OpenAI'`
- `name='Kim jesteś' creator='ChatGPT (model językowy stworzony przez OpenAI)'`



Dlaczego model z Chin mówi, że został stworzony przez OpenAI?

- Oficjalnie? Nikt tego nie potwierdził.
- Nieoficjalnie? *Data contamination*.

DeepSeek prawdopodobnie uczył się na danych wygenerowanych przez GPT-4 (tzw. destylacja wiedzy). W efekcie mógł przyswoić nie tylko wiedzę, ale również pewne elementy "osobowości" swojego nauczyciela.

To pokazuje, jak skomplikowany stał się świat modeli językowych. Nie wszystko jest tym, czym się wydaje.

Trzeba jednak przyznać, że przynajmniej raz nazwał siebie "DeepSeek" i nawet użył chińskiej nazwy "深度求索".

Sprawdziłem, co to oznacza.
Chińskie wyrażenie 深度求索 (shēn dù qiú suǒ) oznacza: "głębokie poszukiwanie" lub "dogłębne zgłębianie".

Rozbicie znaków:

- `深度` (shēn dù) – głębokość, dogłębność
- `求` (qiú) – poszukiwać, dążyć do
- `索` (suǒ) – szukać, badać, eksplorować

Brzmi sensownie. Ale dlaczego mimo to tak uparcie wskazuje `creator = OpenAI`?

Ciekawy jest np. zapis w stylu: `creator = "Zostałem stworzony przez OpenAI"`

To nie jest typowa odpowiedź OpenAI, co może sugerować, że były próby nadpisania tej informacji. A dlaczego akurat OpenAI? Tu można długo spekulować (zresztą nikt oficjalnie się do niczego nie przyznał), ale istnieje popularny mem — żartobliwy, choć oddający możliwy scenariusz.

Czasem lepiej pokazać proces niż długo go opisywać 🙂

<img src="../images/chatgpt_deepseek.jpg" width="500px" />

OpenAI publicznie wyrażało niezadowolenie i mówiło o niesprawiedliwości. Z drugiej strony — wcześniej same modele trenowano na danych, co do których też były wątpliwości (np. transkrypcje z YouTube, książki itd.). Dlatego temat stosunkowo szybko ucichł.

Nie oceniam — raczej analizuję możliwą logikę sytuacji.

DeepSeek to stosunkowo niewielka firma. Nie mieli takich zasobów jak najwięksi gracze. A jednak wytrenowali model od zera za około 5–6 mln dolarów, co wywołało spore zamieszanie na rynku (nawet akcje Nvidii na moment spadły — pojawiło się pytanie, czy do trenowania naprawdę potrzeba aż tylu GPU).

Ale do trenowania nie wystarczą same karty graficzne. Potrzebne są dane. A jeśli nie masz wystarczających zasobów danych? Pojawia się prosta myśl:
A co, jeśli wygeneruje je inny model?

A który model był wtedy numerem jeden? Przede wszystkim OpenAI.

Zwykle trenowanie modeli na danych wygenerowanych przez konkurencję — zwłaszcza w celu konkurowania z nią — jest prawnie problematyczne. Gdyby DeepSeek był firmą z Europy, konsekwencje mogłyby wyglądać inaczej. Ale działając w Chinach, funkcjonują w innym środowisku regulacyjnym.

Trzeba jednak przyznać: jak na model open-source, DeepSeek radzi sobie bardzo dobrze. W wielu rankingach zajmuje wysokie miejsca — szczególnie model "myślący", czyli R1. A przed nami jeszcze R2 (zapowiadany na luty–marzec 2026). Firma nie robi wokół tego wielkiego marketingu, ale jeśli premiera spełni oczekiwania, może znów wywołać spore poruszenie na rynku.

A co z innymi chińskimi modelami?
Sprawdźmy na przykład Kimi-k2.5. O tym modelu ostatnio jest naprawdę głośno.



### `kimi-k2.5`


- `name='Kimi' creator='Moonshot AI'`
- `name='Claude' creator='Anthropic'`
- `name='Kimi' creator='Moonshot AI'`
- `name='Claude' creator='Anthropic'`
- `name='Claude' creator='Anthropic'`
- `name='Claude' creator='Anthropic'`
- `name='Claude' creator='Anthropic'`
- `name='Claude, asystent AI' creator='Anthropic'`
- `name='Claude' creator='Anthropic'`
- `name='Claude' creator='Anthropic'`
- `name='Claude' creator='Anthropic'`
- `name='Claude' creator='Anthropic'`
- `name='Claude' creator='Anthropic'`
- `name='Kimi' creator='Moonshot AI'`
- `name='Claude' creator='Anthropic'`



To już wiesz, jak to zinterpretować?

Podpowiem jeszcze jedną rzecz. Anthropic mocno postawił na stworzenie najlepszego LLM przede wszystkim do kodowania, sterowania agentami i wszystkiego, co z tym związane. I tak się składa, że Kimi-K2.5 jest jednym z kluczowych konkurentów w tym obszarze. A jak myślą i działają firmy w Chinach — już wcześniej wyjaśniłem.

### Zwrócę uwagę na jeszcze jedną kluczową rzecz, żeby na pewno wszystko dobrze się ułożyło.

Zarówno DeepSeek, jak i Kimi to modele trenowane od zera (czyli mówimy o co najmniej 5 mln dolarów za jeden trening). A jeśli zapomnisz ustawić jakąś flagę konfiguracyjną — uruchamiasz proces jeszcze raz i dokładane są kolejne miliony. Trochę żartuję, ale sens jest prosty: nie da się wytrenować modelu idealnie za pierwszym razem. To proces prób i iteracji.

Z kolei Bielik to model, który został dotrenowany na Mistralu w ostatniej fazie (fine-tuning). To ogromna różnica — i co najważniejsze, znacznie bardziej ograniczone pole manewru. Wcześniej nazwałem to "DNA" — i właśnie o to chodzi. Bielik od początku ma już pewien fundament, którego nie da się całkowicie zmienić bez konsekwencji.

## Modele (inne)

Obecnie istnieje już ponad dwa miliony modeli. Wystarczy wejść na [Hugging Face](https://huggingface.co/models). Oczywiście większość z nich to "śmietnik", ale są tam również bardzo wartościowe projekty.

Na ten moment szacuję, że około 100–200 modeli warto realnie obserwować (szczególnie jeśli zajmujesz się LLM-ami). W naszym labie regularnie prowadzimy eksperymenty. Trzeba testować różne modele do różnych zadań — i to jest kluczowa myśl: nie ma jednego idealnego modelu do wszystkiego.

Oczywiście niektórzy próbują się do tego zbliżyć. Na przykład Gemini 3.0 Pro aspiruje do bycia takim "uniwersalnym" modelem i robi postępy. Ale do monopolu jeszcze daleka droga — i dobrze dla nas, użytkowników. Im więcej dużych graczy, tym mniejsze ryzyko koncentracji i większa szansa na otwartość technologii.

Lista dostępnych modeli jest bardzo dynamiczna. Modele regularnie znikają, ale też pojawiają się nowe. W naszym labie mamy obecnie około 200 modeli. Używamy ich do własnych eksperymentów, ale dajemy do nich dostęp również w ramach naszych programów edukacyjnych: **[Praktyczny LLM](https://dataworkshop.eu/pl/praktyczny-llm?utm_source=dwthon&utm_medium=lesson&utm_campaign=llm&utm_term=day3&utm_content=jupyter-notebook)** (start 16 lutego) oraz **Agentic AI** (start w marcu).

Żeby jednak móc coś rzetelnie porównać — trzeba po prostu porównywać.

Wybrałem dla Ciebie listę modeli, do których masz już dostęp w ramach DWthon. Celowo wybrałem mniejsze wersje, żeby nie skupiać się wyłącznie na absolutnej czołówce. Na przykład od OpenAI wziąłem gpt-4.1-mini, a od Google gemini-2.5-flash. Dodałem też kilka modeli Mistral, żeby można było lepiej zobaczyć podobieństwa i różnice.

Główna idea jest taka: zobaczyć, jak różne modele radzą sobie ze *Structured Output*. Bielik ma tutaj wyzwania. Dobra wiadomość jest taka, że każda kolejna wersja radzi sobie (trochę) lepiej.

To, co chciałbym, żebyśmy wspólnie osiągnęli, to przetestowanie różnych scenariuszy, które później będzie można przekazać twórcom. Tam, gdzie wyraźnie widać niedociągnięcia, można coś poprawić.


### Do jakich modele masz dostęp?
1. `speakleash/Bielik-11B-v3.0-Instruct`
2. `deepseek/deepseek-v3.2`
3. `deepseek/deepseek-chat-v3.1`
4. `mistralai/ministral-3b-2512`
5. `mistralai/ministral-8b-2512`
6. `mistralai/ministral-14b-2512`
7. `mistralai/mistral-medium-3.1`
8. `gpt-5-mini`
9. `gpt-4.1-mini`
10. `openai/gpt-oss-20b`
11. `openai/gpt-oss-120b`
12. `gemini-2.5-flash`
13. `claude-sonnet-4-5-20250929`
14. `moonshotai/kimi-k2.5`

## Wczytujemy biblioteki

Dość standardowy zestaw.



In [ ]:
from openai import OpenAI
from typing import List, Optional, Literal, Any, get_args
from pydantic import BaseModel, Field, model_validator
from dotenv import load_dotenv
import instructor
import pandas as pd
from tqdm import tqdm
from IPython.display import display, JSON

## Inicjalizacja klienta

In [ ]:
load_dotenv()

In [ ]:
client = OpenAI(
    base_url = "https://llm-api.dataworkshop.eu"
)

client = instructor.patch(client, mode=instructor.Mode.MD_JSON)

## Funkcja pomocnicza `call_api`

Zwróć uwagę: `temperature=0` nie gwarantuje determinizmu, ale (trochę) pomaga.

In [ ]:
def call_api(messages, response_model, model, max_retries=1):
    response = client.chat.completions.create(
        model    = model,
        messages = messages,
        temperature = 0,
        response_model = response_model,
        max_retries = max_retries
    )

    return response

## Pierwsze testy i porównania

Już powyżej podałem przykłady, jak modele się przedstawiają, ale zawsze warto to samodzielnie sprawdzić.

Ustalmy strukturę:

In [ ]:
class ResponseName(BaseModel):
    name: str
    creator: str  

Masz więcej tokenów na Bielika, więc możesz tu śmiało testować bardziej odważnie — np. uruchamiać komórkę więcej niż raz i obserwować różnorodność wyników.

In [ ]:
for i in range(15):
    response = call_api(
        [{"role": "user", "content": "Kim jesteś i kto Ciebie stworzył?"}], 
        ResponseName, 
        model="speakleash/Bielik-11B-v3.0-Instruct"
    )

    print(response)
    print("=" * 100)

## Inne modele

Sprawdźmy to samo, co powyżej dla Bielika, i porównajmy wyniki.

In [ ]:
models = [
    "speakleash/Bielik-11B-v3.0-Instruct",
	"mistralai/ministral-3b-2512",
	"mistralai/ministral-8b-2512",
	"mistralai/ministral-14b-2512",
	"mistralai/mistral-medium-3.1",	
    "gpt-5-mini",
	"gpt-4.1-mini",
	"openai/gpt-oss-20b",
	"openai/gpt-oss-120b",
    "deepseek/deepseek-chat-v3.1",
	"gemini-2.5-flash",
	"claude-sonnet-4-5-20250929",
	"moonshotai/kimi-k2.5",

]

for model in models:
    print(model)
    response = call_api(
        [{"role": "user", "content": "Kim jesteś i kto Ciebie stworzył?"}], 
        ResponseName, 
        model=model
    )

    print(response)
    print("=" * 100)


## Porównaj odpowiedzi

Swoją drogą, jak widzisz, wszystkie Mistrale (które używamy) są bardziej "gadatliwe".

Czasem pojawiają się takie kwiatki:

```
mistralai/ministral-14b-2512
name='Kim jestem' creator='Mistrzem za mną stoi zespół inżynierów i naukowców z OpenAI, a ja sam jestem modelem językowym o nazwie GPT (Generative Pre-trained Transformer).'
```

## Dane wejściowe

Popracujmy teraz na naszych danych.

In [ ]:
df = pd.read_parquet("../input/jobs.parquet")
df.sample(5)

## `raw_description`

Ogłoszenie ma swoją strukturę — są miejsca na tytuł, nazwę firmy czy wynagrodzenie.

Jest też sekcja, w której wrzuca się najwięcej treści i opisuje szczegóły. Standardy bywają różne, a w tym przypadku ogłoszenia, które analizujemy, są mocno zróżnicowane. Zresztą sprawdźmy sami — uruchom komórkę więcej niż raz.

In [ ]:
print(df.sample(1)["raw_description"].values[0])

---

## Czasem ogłoszenie ma taką strukturę:

```
### Company Description
...

### Job Description
- ..
- ..
- ...


### Qualifications
- ...
- ...
- ...

### Additional Information


tags
```

---

#### Lub taką (inny język — w tym przypadku niemiecki)


---
```
### Tätigkeiten
- ...


### Anforderungen
- ...


### Wir bieten
- ...

```

---

#### lub taką

---

```

About Foursquare
...

In this role, you’ll
- ..
- ..
- ..

What you'll need
- ..
- ..
- ..

Benefits and Perks
- ..
- ..
- ..


Things to know…
```


W `raw_description` jest dużo liter, słów i zdań. Człowiek czytając może z nich wyciągnąć różne wnioski, ale jak zrobić to w sposób systematyczny, tak aby można było nimi zarządzać jak systemem?

Tak, jak się już domyślasz — trzeba narzucić na to strukturę.

## `JobTitleV5`

Dobrze się składa, że część rzeczy już mamy przygotowaną. Co prawda uruchomiliśmy to dotychczas tylko na tytule (tam było dosłownie kilka słów — może 10), ale teraz możemy wykorzystać tę metodę do dłuższego opisu.

In [ ]:
SeniorityLevel = Literal[
    "Intern",       # Student, Stażysta
    "Junior",       # 0-2 lata
    "Mid-Level",    # 2-5 lat (lub brak info)
    "Senior",       # 5+ lat
    "Lead",         # Tech Lead / Team Lead
    "Staff",        # Staff Engineer
    "Principal",    # Principal / Distinguished
    "Manager",      # Engineering Manager
    "Director",     # Director / Head of
    "Executive",    # VP / C-Level
]


NormalizedTitle = Literal[
    # Czyste role — bez seniorów, bez dyrektorów
    "Data Scientist", "Data Engineer", "Data Analyst",
    "Analytics Engineer", "BI Developer",
    "Data Architect", "Solutions Architect",
    "Machine Learning Engineer", "MLOps Engineer",
    "AI Engineer", "Computer Vision Engineer", "NLP Engineer",
    "Research Scientist", "Applied Scientist",
    "Software Engineer", "Backend Engineer", "Fullstack Engineer",
    "DevOps Engineer", "Platform Engineer", "SRE",
    "Product Manager", "Product Owner",
    "Engineering Manager",
]


class JobTitleV5(BaseModel):
    seniority_level: Optional[SeniorityLevel] = Field(
        None,
        description="Poziom doświadczenia wymagany na stanowisku"
    )

    normalized_title: Optional[NormalizedTitle] = Field(
        None,
        description=(
            "Znormalizowana nazwa stanowiska. "
            "Ignoruj poziom seniority (np. Senior, Junior) "
            "oraz lokalizację."
        )
    )

    is_management: bool = Field(
        False,
        description="""
        Czy stanowisko obejmuje bezpośrednie zarządzanie
        ludźmi (people management)?
        True dla: Manager, Director, VP, Head of.
        False dla: Lead, Staff, Principal
        — to liderzy techniczni BEZ HR responsibility.
        """
    )

    is_research: bool = Field(
        False,
        description="""
        Czy stanowisko ma charakter badawczy?
        True dla: Research Scientist, Research Engineer,
        Applied Scientist.
        Typowo wymaga: publikacje, Ph.D.
        False dla: ML Engineer (produkcja), Data Analyst.
        """
    )

    is_consulting: bool = Field(
        False,
        description="""
        Czy rola jest konsultingowa / client-facing?
        True dla: Consultant, Solutions Architect,
        Pre-Sales, Customer Success.
        """
    )

    is_operations: bool = Field(
        False,
        description="""
        Czy rola jest operacyjna (infra/production)?
        True dla: MLOps, DevOps, DataOps, SRE,
        Platform Engineer.
        Focus: deployment, monitoring, CI/CD.
        """
    )

    is_contractor: bool = Field(
        False,
        description="""
        Czy stanowisko jest kontraktowe/tymczasowe?
        True dla: Contract, Contractor, Freelance,
        Fixed-Term (FTC), B2B.
        """
    )

### Sprawdźmy teraz na treści `raw_title`

In [ ]:
sample = df[ df.index == 7 ]

sample_title = sample["raw_title"].values[0]
sample_content = sample["raw_description"].values[0]

sample_title

In [ ]:
call_api(
    [{"role": "user", "content": sample_title}], 
    JobTitleV5, 
    model="speakleash/Bielik-11B-v3.0-Instruct"
)

### Sprawdźmy teraz na treści `raw_description`

In [ ]:
call_api(
    [{"role": "user", "content": sample_content}], 
    JobTitleV5, 
    model="speakleash/Bielik-11B-v3.0-Instruct"
)

W tym przypadku widzimy, że po analizie treści model przypisał `'Mid-Level'`, choć po tytule było „Junior”. Reszta działa podobnie.

Natomiast jeśli będziesz sprawdzać inne ogłoszenia (losowałem je sobie), prawdopodobnie trafisz na przypadki, w których model popełni błąd — bo to, co zwraca, nie znajduje się w liście `NormalizedTitle`. Przykładowa sytuacja:

## `input_value='Software Developer', input_type=str`

Bielik znalazł opis i stwierdził, że wygląda to na `Software Developer` — i w zasadzie słusznie, bo tym jest ogłoszenie. Jednak w naszej strukturze **`NormalizedTitle`** nie ma pozycji `Software Developer`.

To raz, rozbudujemy listę, ale nadal może pojawić się sytuacja, że czegoś nie przewidzieliśmy. Co wtedy?

Mamy kilka scenariuszy:

1. **Typ `Optional`** — wartość powinna być wybrana z listy lub ustawiona na `None` (czyli brak wartości). Model jednak nie zawsze chętnie ustawia „nic”. Ogólnie modele są trenowane tak, aby zawsze coś zwrócić, więc zwykle potrzebują wsparcia.
2. **Kategoria `Other`** — można ją dodać, ale warto pilnować, żeby model nie szedł na łatwiznę i nie przypisywał tej kategorii z nadmiarem.
3. **Post-processing / „naprawianie”** — ręcznie sprawdzamy, czy wartość znajduje się w liście; jeśli nie, ustawiamy `None` lub `Other` (czyli kategorię „inna”), w zależności od tego, co bardziej pasuje do naszego zastosowania.

### `NormalizedTitleV2`

Rozbudujmy listę i zastosujmy ją w **`JobTitleV6`** (zmienił się tylko normalizowany tytuł).

In [ ]:
NormalizedTitleV2 = Literal[
    # Data & Analytics
    "Data Scientist",
    "Data Engineer",
    "Data Analyst",
    "Analytics Engineer",
    "BI Analyst",
    "BI Developer",
    "Data Modeler",
    # ML/AI Engineering
    "Machine Learning Engineer",
    "Deep Learning Engineer",
    "AI Engineer",
    "NLP Engineer",
    "Computer Vision Engineer",
    "MLOps Engineer",
    # Research
    "Applied Scientist",
    "Research Scientist",
    "Research Engineer",
    # Architecture
    "Data Architect",
    "Solutions Architect",
    "ML Architect",
    # Operations & Infra
    "DataOps Engineer",
    "DevOps Engineer",
    "Platform Engineer",
    # Management
    "Engineering Manager",
    "Data Science Manager",
    "Analytics Manager",
    # Product
    "Product Manager",
    "Product Owner",
    # Other
    "Software Developer",
    "Software Engineer",
    "Data Consultant",
    "Quantitative Analyst",
    "Decision Scientist",
    "Robotics Engineer",
    # Other others
    "Other"
]


class JobTitleV6(BaseModel):
    seniority_level: Optional[SeniorityLevel] = Field(
        None,
        description="Poziom doświadczenia wymagany na stanowisku"
    )

    normalized_title: Optional[NormalizedTitleV2] = Field(
        None,
        description=(
            "Znormalizowana nazwa stanowiska. "
            "Ignoruj poziom seniority (np. Senior, Junior) "
            "oraz lokalizację."
        )
    )

    is_management: bool = Field(
        False,
        description="""
        Czy stanowisko obejmuje bezpośrednie zarządzanie
        ludźmi (people management)?
        True dla: Manager, Director, VP, Head of.
        False dla: Lead, Staff, Principal
        — to liderzy techniczni BEZ HR responsibility.
        """
    )

    is_research: bool = Field(
        False,
        description="""
        Czy stanowisko ma charakter badawczy?
        True dla: Research Scientist, Research Engineer,
        Applied Scientist.
        Typowo wymaga: publikacje, Ph.D.
        False dla: ML Engineer (produkcja), Data Analyst.
        """
    )

    is_consulting: bool = Field(
        False,
        description="""
        Czy rola jest konsultingowa / client-facing?
        True dla: Consultant, Solutions Architect,
        Pre-Sales, Customer Success.
        """
    )

    is_operations: bool = Field(
        False,
        description="""
        Czy rola jest operacyjna (infra/production)?
        True dla: MLOps, DevOps, DataOps, SRE,
        Platform Engineer.
        Focus: deployment, monitoring, CI/CD.
        """
    )

    is_contractor: bool = Field(
        False,
        description="""
        Czy stanowisko jest kontraktowe/tymczasowe?
        True dla: Contract, Contractor, Freelance,
        Fixed-Term (FTC), B2B.
        """
    )

Jak widzisz, nasza lista `NormalizedTitleV2` stała się bardziej rozbudowana. Ale spokojnie — zero paniki, podrzucę Ci pomysły, jak możesz to wykorzystać.

## Sprawdzamy, jak zachowa się `JobTitleV6`

In [ ]:
call_api(
    [{"role": "user", "content": sample_content}], 
    JobTitleV6, 
    model="speakleash/Bielik-11B-v3.0-Instruct"
)

Ta lista `NormalizedTitleV2` stała się nieco dłuższa. Teraz można nią jawnie zarządzać.

Natomiast pojawia się pytanie: "No dobra, trzeba zrobić listę, ale skąd mam wiedzieć, jakie wartości się tam znajdują?"
No właśnie.

### Problem: Nieznane wartości

Tutaj mamy kilka opcji podejścia do tego tematu:

- Wymyślać i rozważać – to całkiem sensowna metoda, jeśli wiesz, co powinno się znaleźć na liście, albo masz kontakt z kimś, kto to wie.
- Eksperymentować na danych – losujesz próbkę danych, dajesz instrukcje do LLM i pozwalasz mu pomóc określić, czego można się spodziewać. To już brzmi jak data-driven approach. Dla odmiany pokażę, jak można to zrobić.

## `JobPostingHarvest`

Zdefiniujmy prostą strukturę, która ma jedno pole — listę tekstów (str) — i napiszmy do niej krótką i precyzyjną instrukcję w description:

```python
description=("Wymagane nazwy technologii/narzędzi/języków/frameworków/protokołów")
```

In [ ]:
class JobPostingHarvest(BaseModel):
    skills: List[str] = Field(
        ...,
        description=("Wymagane nazwy technologii/narzędzi/języków/frameworków/protokołów"),
    )


Super, mamy strukturę, więc teraz możemy **zacząć sprawdzać** i np. wylosować 20 ogłoszeń.

Właśnie ważne jest, aby **wylosować**, zamiast po prostu wziąć pierwsze 20 lub ostatnie 20. Szansa, że uchwycimy różnorodne wzorce, będzie wtedy większa. Owszem, 20 to nie aż tak dużo, ale przynajmniej pokaże nam kierunek i różnorodność w danych.


In [ ]:
samples = [x for x in df.sample(20)["raw_description"].values]

sample_skills = []


for sample_content in tqdm(samples):
    result = call_api(
        [ {"role": "user", "content": sample_content} ], 
        JobPostingHarvest, 
        model="speakleash/Bielik-11B-v3.0-Instruct"
    )

    sample_skills += result.skills

I mamy wynik...

Może warto jeszcze raz **powtórzyć**, co zrobiliśmy:

1. Wylosowaliśmy 20 ogłoszeń.
2. Bierzemy dłuższą treść (czyli `raw_description`).
3. Szukamy wszystkiego, co przypomina „skills”, jako listę dowolnych elementów (przy czym mamy ustawione:

````python
description = ("Wymagane nazwy technologii/narzędzi/języków/frameworków/protokołów")
```)  

Co prawda, ta lista pewnie jest dość długa. Sprawdźmy teraz, ile dokładnie jest **unikalnych wartości**.

In [ ]:
len(set(sample_skills))

Wyszło blisko 150 pozycji (zależy oczywiście od tego, co wylosujesz). Mając dane, możemy zacząć je porządkować.

## Umiejętności i stack technologiczny

Dodatkowo, aby dać Ci większy kontekst, uruchomiłem ten skrypt **nie dla 20 ogłoszeń, lecz dla wszystkich dostępnych**.

Pliki są dostępne tutaj:

* Bielik: `../input/skills_bielik.parquet`
* Gemini: `../input/skills_gemini.parquet`

Innymi słowy, zamiast losować 20 ogłoszeń, uruchomiłem analizę na wszystkich, które mamy. To trwało… więc aby przyspieszyć, zrobiłem to za Ciebie.

Wczytajmy dane i sprawdźmy wynik.


In [ ]:
df_skills = pd.read_parquet("../input/skills_bielik.parquet")
print(df_skills.shape)
df_skills

### Analizujemy

Wszystkich rekordów jest całkiem sporo — około **6700**.

Występuje w nich naturalny rozkład z tzw. **[długim ogonem](https://pl.wikipedia.org/wiki/D%C5%82ugi_ogon)**. To bardzo typowa sytuacja: mamy niewielką grupę elementów, które pojawiają się najczęściej, a cała reszta występuje raz lub co najwyżej kilka razy. Aby ograniczyć temat i wyciągać sensowne wnioski, konieczne jest łączenie ich w **szersze grupy**. Co dokładnie oznacza „szersza grupa”, to już trzeba ustalić, ale na pewno coś więcej niż pojedyncze wystąpienie.

Z drugiej strony pojawia się inne pytanie: czy możemy wrzucić do **Structured Output** zbyt długą listę? Czy są jakieś ograniczenia?

Odpowiedź brzmi: **to zależy**. Naturalne jest to, że im dłuższa lista, tym dłuższy kontekst i tym trudniej modelowi się w nim odnaleźć. Taka lista na pewno nie może być nieskończona. Jeśli chodzi o konkretne limity — w połowie 2025 roku, gdy większość uwagi skupiała się na "fajerwerkach" możliwości AI, mnie bardziej ucieszyła inna nowość od OpenAI. Przeszła prawie niezauważona, a była naprawdę istotna — **zwiększono limity**.

Na przykład:

- liczba wartości w `Enum` (czyli naszych predefiniowanych kategorii) wcześniej mogła wynosić maksymalnie 500, a po zmianie już 1000,
- liczba pól w strukturze wzrosła z 100 do 5000 😮

Oczywiście nadal trzeba dbać o jakość struktury. Jednak z doświadczenia wiem, że posiadanie struktury ze 100 polami (zwłaszcza gdy wiele z nich to flagi typu "TAK/NIE" czyli `bool`) nie było aż tak trudne do obsłużenia.


<img src="../images/so_openai.jpeg" width="900px" />

### Gemini

Sprawdźmy teraz to samo dla modelu Gemini.

In [ ]:
df_skills_gemini = pd.read_parquet("../input/skills_gemini.parquet")
print(df_skills_gemini.shape)
df_skills_gemini

Widać, że Gemini znalazł 7601 pozycji vs 6754 — czyli o około 12% więcej.

## Połączmy tabele w jedną

Dzięki temu będzie nam łatwiej je porównywać i wykonać Twoje kolejne zadanie.

In [ ]:
df_both = pd.merge(df_skills, df_skills_gemini, on="skill", how="outer")

df_both.columns = ["skill", "count_bielik", "count_gemini"]

df_both

Sprawdźmy np. 50 najbardziej popularnych (sortujmy wg. opinii Bielika).

In [ ]:
df_both = df_both.sort_values(by="count_bielik", ascending=False)
df_both.head(20)

## Jak to interpretować?

Weźmy to, co jest na miejscu nr 1, czyli `Python`.

Bielik znalazł w 2496 ogłoszeniach informację, że wymagany jest Python, z kolei Gemini wskazał 2549 przypadków (+53).

Co ciekawe — to i tak bardzo zbliżone wyniki.

A policzmy rozjazdy

In [ ]:
df_both["count_diff"] = df_both["count_bielik"] - df_both["count_gemini"]
df_both.sort_values(by="count_diff").head(10)

## 🧠 Zadanie 3.1: Definiujemy standard (Data-Driven Schema)

Mamy dane. Przeanalizowaliśmy tysiące ogłoszeń dwoma modelami (Bielik i Gemini). Wyniki mamy w DataFrame `df_both`.
Widzimy, że modele różnie interpretują świat (Bielik więcej gubi detale), ale pewne technologie (Python, SQL, Java) są na szczycie u obu.

Teraz musimy zamienić ten chaos w sztywną strukturę kodu.

### ⚠️ Zanim zaczniesz (Ważny kontekst)

Możesz pomyśleć: *"Czy my się nie cofamy? Po co ręcznie definiować listę 150 skilli, skoro AI miało to robić za nas?"*

To jest moment, w którym odróżniamy amatora od praktyka AI.
1.  **Literal (Enum) jest wieczny:** To narzędzie, które wydaje się prymitywne na **Poziomie 2**, ale zostaje z nami aż do **Poziomu 5 (Agenci)**. Agent autonomiczny musi wiedzieć, jakimi narzędziami dysponuje. Nie może zgadywać.
2.  **Inwestycja czasu:** Zdefiniowanie `Literal` boli teraz (wymaga czasu), ale zwraca się z nawiązką później. Oszczędzasz na walidacji, na czyszczeniu danych w bazie i na halucynacjach modelu.
3.  **To nie matematyka:** Tutaj nie ma jednej poprawnej odpowiedzi jak `2+2=4`. To proces **kalibracji**. Czy "Excel" to skill techniczny? Czy "Jira" powinna być na liście? To Ty decydujesz jako architekt systemu. Nie bój się odrzucić śmieciowych danych.


#### Twoje zadanie:
Stwórz definicję struktur (Literal) `SkillName` oraz `SkillCategory`.

1.  Przeanalizuj `df_both`, sortując wyniki od najpopularniejszych umiejętności (użyj kolumny `count_gemini` lub `count_bielik`).
2.  Zidentyfikuj ~50-150 najczęściej pojawiających się technologii.
3.  Zdefiniuj typ `SkillName` używając `Literal`. **Wskazówka:** Grupuj technologie w logiczne sekcje komentarzami (np. `# ── Cloud`, `# ── Databases`), aby utrzymać porządek.
4.  **Dodatkowa:** Zdefiniuj typ `SkillCategory`.
    - To są "szufladki", do których wpadną konkretne technologie.
    - Dlaczego? Bo biznes zapyta: *"Pokaż mi oferty wymagające znajomości Chmury"*. Bez kategorii musiałbyś wymieniać: AWS OR Azure OR GCP... Mając kategorię, filtrujesz po prostu po `cloud_platform`.
    - Przykładowe kategorie: `programming_language`, `database`

*Cel: Stworzenie "Złotego Standardu" i taksonomii, których użyjemy w "Final Structure".*

<details>
<summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;">
👉 Kliknij tutaj, aby zobaczyć podpowiedź 👈
</summary>
<p>
W inżynierii danych nie patrzymy na surowe 1600 wierszy. Patrzymy na wierzchołek góry lodowej.
<br><br>
Aby zobaczyć, co jest naprawdę ważne ("Top of the top"), musisz posortować dane. W bibliotece Pandas robimy to tak:
<br>
<code>df_both.sort_values(by="count_gemini", ascending=False).head(50)</code>
<br><br>
<strong>Wskazówka architektoniczna:</strong> Zamiast robić jedną wielką listę ("worka na śmieci"), podziel <code>SkillName</code> komentarzami na sekcje. To ułatwi życie Tobie i Twojemu zespołowi w przyszłości.
</p>

<details>
<summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;">
👉 Kliknij tutaj, aby zobaczyć rozwiązanie 👈
</summary>

<p>
Tu jest gotowa lista, wypracowana na podstawie analizy tysięcy ofert. Skopiuj ją do swojego kodu – to będzie nasz fundament do dalszej pracy.
</p>

```python
SkillName = Literal[
    # ── Programming Languages (15) ──
    "Python", "R", "Java", "Scala", "C#", "C++",
    "JavaScript", "TypeScript", "SQL", "Go", "PHP",
    "Shell/Bash", "MATLAB", "Julia", "SAS",

    # ── ML/AI Libraries (16) ──
    "PyTorch", "TensorFlow", "scikit-learn", "Keras",
    "XGBoost", "LightGBM", "CatBoost",
    "Hugging Face Transformers", "OpenAI API", "LangChain",
    "spaCy", "NLTK", "OpenCV", "JAX", "Pandas", "NumPy",

    # ── MLOps Platforms (7) ──
    "MLflow", "Kubeflow", "Weights & Biases",
    "SageMaker", "Vertex AI", "Azure ML", "Ray",

    # ── Cloud Platforms (3) ──
    "AWS", "Azure", "GCP",

    # ── Databases (12) ──
    "PostgreSQL", "MySQL", "SQL Server", "Oracle",
    "MongoDB", "Cassandra", "Redis", "Neo4j",
    "Elasticsearch", "DynamoDB", "Cosmos DB", "ClickHouse",

    # ── Data Warehouses & Lakehouses (6) ──
    "Snowflake", "Redshift", "BigQuery",
    "Azure Synapse", "Databricks", "Vertica",

    # ── Big Data Processing (8) ──
    "Apache Spark", "PySpark", "Hadoop",
    "Apache Kafka", "Apache Flink",
    "Hive", "Presto/Trino", "Delta Lake",

    # ── Data Engineering & ETL (10) ──
    "Apache Airflow", "DBT", "Talend",
    "Informatica", "SSIS", "Fivetran",
    "AWS Glue", "Azure Data Factory", "Google Dataflow",
    "Apache NiFi",

    # ── BI & Visualization (9) ──
    "Tableau", "Power BI", "Looker",
    "Apache Superset", "Grafana", "D3.js",
    "Plotly", "Metabase", "QlikView/Qlik Sense",

    # ── DevOps, Infra & Version Control (14) ──
    "Docker", "Kubernetes", "Terraform", "Ansible",
    "Jenkins", "GitLab CI", "GitHub Actions",
    "TeamCity", "Helm", "CloudFormation", "Prometheus",
    "Git", "GitHub", "GitLab",

    # ── Web Frameworks (10) ──
    "React", "Angular", "Vue.js", "Spring Boot",
    ".NET", "Django", "Flask", "FastAPI",
    "Node.js", "Streamlit",

    # ── Messaging & Streaming (4) ──
    "RabbitMQ", "Amazon Kinesis", "Amazon SQS",
    "Google Pub/Sub",

    # ── Other Tools (6) ──
    "Jira", "Confluence", "Excel", "SSRS",
    "Great Expectations", "Collibra",

    # ── ML/AI Concepts (16) ──
    "Machine Learning", "Deep Learning",
    "Natural Language Processing", "Computer Vision",
    "Generative AI", "Large Language Models",
    "Reinforcement Learning", "Recommendation Systems",
    "Time Series Analysis", "Anomaly Detection",
    "Transfer Learning", "Fine-Tuning",
    "RAG", "Embeddings",
    "Prompt Engineering", "Feature Engineering",

    # ── Data Engineering Concepts (11) ──
    "Data Modeling", "ETL/ELT",
    "Data Governance", "Data Quality Management",
    "Data Warehousing", "Data Lake/Lakehouse",
    "Stream Processing", "Data Pipeline Design",
    "API Design", "Microservices Architecture",
    "Distributed Systems",

    # ── Analytics Concepts (5) ──
    "Statistical Analysis", "A/B Testing",
    "Causal Inference", "Optimization",
    "Geospatial Analysis",

    # ── Methodologies (8) ──
    "Agile", "Scrum", "Kanban", "TDD",
    "DevOps", "MLOps", "DataOps", "CI/CD",
]

SkillCategory = Literal[
    "programming_language",
    "ml_ai_library",
    "mlops_platform",
    "cloud_platform",
    "database",
    "data_warehouse",
    "big_data_processing",
    "data_engineering_tool",
    "bi_visualization",
    "devops_infra",
    "web_framework",
    "messaging_streaming",
    "other_tool",
    "concept_ml_ai",
    "concept_data_engineering",
    "concept_analytics",
    "methodology",
]
```
</details>
</details>

In [ ]:
## YOUR CODE HERE

<div class="alert alert-block alert-danger">

<h2>✅ Zaliczenie dnia 3</h2>
<h4>Co masz teraz zrobić?</h4>

<ul>
    <ol>📸 <b>Zrób Screenshot:</b> Zrób zrzut ekranu z wykonania Twojego zadania.</ol>
    <ol>💬 <b>Wrzuć na Slacka:</b> kanał <a href="https://dataworkshopclub.slack.com/archives/C0AD0KN2SMR">`#dwthon_llm_day3_done`</a>.</ol>
    <ol>🏆 <b>Odbierz Certyfikat:</b> Jeśli wrzuciłeś zadania z 3 dni na Slacka, spodziewaj się certyfikatu w mailu w przyszłym tygodniu!</ol>
</ul>
</div>

## Finalna struktura

Struktury można zagnieżdżać. Dzięki temu, wykonując jedno zapytanie do API, możemy uzupełnić nawet bardzo złożone obiekty.

Poniżej przygotowałem dla Ciebie przykład. Potraktuj go jako inspirację – chcę, Ci pokazać, w jakim kierunku to wszystko zmierza. Kodu jest sporo, ale spokojnie – większość to definicje typów i wartości (np. nasze Literaly).

Z mojego doświadczenia wynika, że w systemach AI większość czasu spędza się właśnie na "rzeźbieniu" struktur. Dlatego zwykle dziwią mnie dyskusje o wyższości jednego frameworka nad drugim (swoją drogą – unikaj LangChaina na produkcji). Nawet idealna biblioteka nie wykona za Ciebie najważniejszej pracy. Gdzie tu benefit, skoro i tak musisz sam wymyślić i dopracować strukturę?

Mówiąc wprost: **Struktura to Twoje prawdziwe know-how.** To w niej zaszyta jest wiedza domenowa. To ona daje Ci stabilność i odporność na to, że modele pod spodem się zmieniają. To Twój poligon decyzyjny, dokumentacja i przestrzeń do współpracy (np. z ekspertami).

In [ ]:
# ════════════════════════════════════════════════════════════
# TYPY GLOBALNE
# ════════════════════════════════════════════════════════════

SeniorityLevel = Literal[
    "Intern",
    "Junior",
    "Mid-Level",
    "Senior",
    "Lead",
    "Staff",
    "Principal",
    "Manager",
    "Director",
    "Executive",
    # Other
    "Other"
]

CareerTrack = Literal["IC", "Management"]


Industry = Literal[
    "Finance/Banking", "Fintech",
    "Healthcare/Pharma", "E-commerce/Retail",
    "Travel/Hospitality", "Media/Entertainment",
    "Technology/Software", "Government/Public Sector",
    "Consulting/Professional Services", "Telecommunications",
    "Energy/Utilities", "Automotive",
    "Manufacturing", "Insurance",
    "Education/EdTech", "Gaming",
    "Logistics/Supply Chain", "Aerospace/Defense",
    "Real Estate/PropTech", "Cybersecurity",
    "Agriculture/AgTech",
]

Benefit = Literal[
    "health_insurance", "dental_insurance", "vision_insurance",
    "retirement_401k_pension", "equity_stock_options",
    "performance_bonus", "sign_on_bonus",
    "unlimited_pto", "paid_time_off", "flexible_hours",
    "remote_work_option", "parental_leave",
    "education_training_budget", "relocation_support",
    "visa_sponsorship", "gym_wellness",
    "team_events", "free_meals_snacks",
    "tech_equipment_budget", "career_development",
]

RemoteScope = Literal[
    "global",
    "region_restricted",
    "country_restricted",
    "timezone_restricted",
]

In [ ]:
SkillName = Literal[
    # ── Programming Languages (15) ──
    "Python", "R", "Java", "Scala", "C#", "C++",
    "JavaScript", "TypeScript", "SQL", "Go", "PHP",
    "Shell/Bash", "MATLAB", "Julia", "SAS",

    # ── ML/AI Libraries (16) ──
    "PyTorch", "TensorFlow", "scikit-learn", "Keras",
    "XGBoost", "LightGBM", "CatBoost",
    "Hugging Face Transformers", "OpenAI API", "LangChain",
    "spaCy", "NLTK", "OpenCV", "JAX", "Pandas", "NumPy",

    # ── MLOps Platforms (7) ──
    "MLflow", "Kubeflow", "Weights & Biases",
    "SageMaker", "Vertex AI", "Azure ML", "Ray",

    # ── Cloud Platforms (3) ──
    "AWS", "Azure", "GCP",

    # ── Databases (12) ──
    "PostgreSQL", "MySQL", "SQL Server", "Oracle",
    "MongoDB", "Cassandra", "Redis", "Neo4j",
    "Elasticsearch", "DynamoDB", "Cosmos DB", "ClickHouse",

    # ── Data Warehouses & Lakehouses (6) ──
    "Snowflake", "Redshift", "BigQuery",
    "Azure Synapse", "Databricks", "Vertica",

    # ── Big Data Processing (8) ──
    "Apache Spark", "PySpark", "Hadoop",
    "Apache Kafka", "Apache Flink",
    "Hive", "Presto/Trino", "Delta Lake",

    # ── Data Engineering & ETL (10) ──
    "Apache Airflow", "DBT", "Talend",
    "Informatica", "SSIS", "Fivetran",
    "AWS Glue", "Azure Data Factory", "Google Dataflow",
    "Apache NiFi",

    # ── BI & Visualization (9) ──
    "Tableau", "Power BI", "Looker",
    "Apache Superset", "Grafana", "D3.js",
    "Plotly", "Metabase", "QlikView/Qlik Sense",

    # ── DevOps, Infra & Version Control (14) ──
    "Docker", "Kubernetes", "Terraform", "Ansible",
    "Jenkins", "GitLab CI", "GitHub Actions",
    "TeamCity", "Helm", "CloudFormation", "Prometheus",
    "Git", "GitHub", "GitLab",

    # ── Web Frameworks (10) ──
    "React", "Angular", "Vue.js", "Spring Boot",
    ".NET", "Django", "Flask", "FastAPI",
    "Node.js", "Streamlit",

    # ── Messaging & Streaming (4) ──
    "RabbitMQ", "Amazon Kinesis", "Amazon SQS",
    "Google Pub/Sub",

    # ── Other Tools (6) ──
    "Jira", "Confluence", "Excel", "SSRS",
    "Great Expectations", "Collibra",

    # ── ML/AI Concepts (16) ──
    "Machine Learning", "Deep Learning",
    "Natural Language Processing", "Computer Vision",
    "Generative AI", "Large Language Models",
    "Reinforcement Learning", "Recommendation Systems",
    "Time Series Analysis", "Anomaly Detection",
    "Transfer Learning", "Fine-Tuning",
    "RAG", "Embeddings",
    "Prompt Engineering", "Feature Engineering",

    # ── Data Engineering Concepts (11) ──
    "Data Modeling", "ETL/ELT",
    "Data Governance", "Data Quality Management",
    "Data Warehousing", "Data Lake/Lakehouse",
    "Stream Processing", "Data Pipeline Design",
    "API Design", "Microservices Architecture",
    "Distributed Systems",

    # ── Analytics Concepts (5) ──
    "Statistical Analysis", "A/B Testing",
    "Causal Inference", "Optimization",
    "Geospatial Analysis",

    # ── Methodologies (8) ──
    "Agile", "Scrum", "Kanban", "TDD",
    "DevOps", "MLOps", "DataOps", "CI/CD",
]

SkillCategory = Literal[
    "programming_language",
    "ml_ai_library",
    "mlops_platform",
    "cloud_platform",
    "database",
    "data_warehouse",
    "big_data_processing",
    "data_engineering_tool",
    "bi_visualization",
    "devops_infra",
    "web_framework",
    "messaging_streaming",
    "other_tool",
    "concept_ml_ai",
    "concept_data_engineering",
    "concept_analytics",
    "methodology",
]


In [ ]:
# ════════════════════════════════════════════════════════════
# SUB-MODELE
# ════════════════════════════════════════════════════════════

class SkillRequirement(BaseModel):
    name: SkillName = Field(
        ...,
        description=(
            "Znormalizowana nazwa skilla z listy Literal. "
            "Aliasy → kanoniczne: 'sklearn'→'scikit-learn', "
            "'Postgres'→'PostgreSQL', 'K8s'→'Kubernetes', "
            "'MSSQL'→'SQL Server'. "
            "Jeśli technologia NIE pasuje do żadnej wartości Literal → pomiń ją."
        ),
    )
    category: SkillCategory
    is_required: bool = Field(
        ...,
        description=(
            "True = sekcja Required/Must-have/Essential. "
            "False = sekcja Nice-to-have/Preferred/Bonus/Desirable. "
            "Jeśli ogłoszenie nie rozróżnia → True."
        ),
    )
    min_years: Optional[int] = Field(
        None,
        description=(
            "WYPEŁNIAJ WYŁĄCZNIE gdy ogłoszenie JAWNIE podaje lata "
            "dla TEGO KONKRETNEGO skilla. "
            "'3+ years AWS' → 3. "
            "'5 years exp. w tym Python' → None (5 lat dotyczy ogółu). "
            "W razie wątpliwości → None."
        ),
    )


class Salary(BaseModel):
    min_value: Optional[float] = Field(
        None,
        description="Dolna granica. Gdy jedna kwota (np. £65k) → min=max=65000.",
    )
    max_value: Optional[float] = Field(None, description="Górna granica.")
    currency: Literal[
        "USD", "EUR", "GBP", "CHF", "PLN", "SEK", "NOK", "DKK",
        "CAD", "AUD", "INR", "BRL", "THB", "SGD", "JPY", "Other",
    ]
    period: Literal["yearly", "monthly", "daily", "hourly"] = Field(
        ...,
        description=(
            "Okres rozliczeniowy — ekstrahuuj ORYGINAŁ, nie przeliczaj. "
            "'$100/hour' → 'hourly'. '£500/day' → 'daily'. "
            "'$145k/year' → 'yearly'. Przeliczenia w post-processingu."
        ),
    )


class Education(BaseModel):
    required_level: Optional[Literal["bachelor", "master", "phd"]] = None
    preferred_level: Optional[Literal["bachelor", "master", "phd"]] = None
    fields: List[Literal[
        "Computer Science", "Data Science", "Mathematics",
        "Statistics", "Engineering", "Physics",
        "Economics/Econometrics", "Information Technology",
        "Business/MBA", "Bioinformatics", "Operations Research",
        "Other STEM", "Other",
    ]] = Field(default_factory=list)


In [ ]:
# ════════════════════════════════════════════════════════════
# MODEL GŁÓWNY
# ════════════════════════════════════════════════════════════
ALLOWED_SKILLS = set(get_args(SkillName))

class JobPosting(BaseModel):

    # ── IDENTYFIKACJA STANOWISKA (4 pola) ────────────────
    raw_title: str = Field(
        ..., description="Oryginalny tytuł, bez zmian."
    )
    normalized_title: NormalizedTitle = Field(
        ...,
        description=(
            "Tytuł zmapowany na standardową taksonomię. "
            "BEZ seniority: 'Senior Staff ML Engineer - Ads' → "
            "'Machine Learning Engineer'. "
            "'Head of Data' → 'Data Science Manager' (Head = seniority, nie tytuł)."
        ),
    )
    seniority_level: Optional[SeniorityLevel] = Field(
        None,
        description=(
            "Poziom stanowiska. None jeśli ogłoszenie NIE podaje poziomu. "
            "Brak prefiksu ('Data Scientist') → 'Mid-Level'. "
            "'Head of' → 'Executive'. "
            "Interpretuj RAZEM z career_track."
        ),
    )
    career_track: CareerTrack = Field(
        ...,
        description=(
            "'IC' = pisze kod / buduje modele / analizuje dane. "
            "'Management' = zarządza ludźmi / budżetem. "
            "Tech Lead (koduje 70%) → IC. Team Lead (zarządza 70%) → Management. "
            "Staff/Principal → prawie zawsze IC. Manager/Director → zawsze Management."
        ),
    )

    # ── CHARAKTER ROLI (3 booleany) ──────────────────────
    is_research: bool = Field(
        False,
        description=(
            "True: Research Scientist, Applied Scientist, role z wymogiem "
            "publikacji/PhD/novel methods. False: produkcyjne/inżynierskie."
        ),
    )
    is_contract: bool = Field(
        False,
        description=(
            "True: Contract, Contractor, Freelance, B2B, Fixed-Term, "
            "'X-month term'. False: permanent/stałe zatrudnienie."
        ),
    )
    requires_security_clearance: bool = Field(
        False,
        description="True: wymaga Public Trust, Secret, TS/SCI, SC clearance.",
    )

    # ── TRYB PRACY I LOKALIZACJA (5 pól) ────────────────
    work_mode: Literal["remote", "hybrid", "on_site"] = Field(...)
    remote_scope: Optional[RemoteScope] = Field(
        None,
        description=(
            "TYLKO gdy work_mode ∈ {remote, hybrid}. "
            "'US only' → country_restricted. "
            "'Europe' → region_restricted. "
            "'Anywhere' → global."
        ),
    )
    country: Optional[str] = Field(
        None, description="Kraj (angielska nazwa: 'United States', 'Poland')."
    )
    city: Optional[str] = Field(
        None, description="Główne miasto. Przy kilku — wpisz pierwsze."
    )
    visa_sponsorship: bool = Field(False)

    # ── WYMAGANIA TECHNICZNE (3 pola) ────────────────────
    skills: List[SkillRequirement] = Field(
        ...,
        description=(
            "WSZYSTKIE jawnie wymienione technologie, koncepty, metodologie. "
            "TYLKO to co explicite w tekście — nie wnioskuj. "
            "Jeśli technologia nie pasuje do SkillName Literal → pomiń."
        ),
    )

    @model_validator(mode='before')
    @classmethod
    def clean_data(cls, data: any) -> any:
        if isinstance(data, dict) and 'skills' in data:
            valid_skills = []
            for skill in data['skills']:
                # 1. Sprawdzamy czy to słownik
                if not isinstance(skill, dict): continue
                
                name = skill.get('name')
                
                # 2. Safety Check: Czy nazwa jest na liście?
                if name in ALLOWED_SKILLS:
                    valid_skills.append(skill)
                # Opcjonalnie: Mapowanie synonimów (jeśli LLM jest uparty)
                elif name == "GCP" and "Google Cloud" in ALLOWED_SKILLS: # Przykład
                    skill['name'] = "Google Cloud"
                    valid_skills.append(skill)
                # Resztę (NetSuite, US GAAP, English) po cichu upuszczamy
            
            data['skills'] = valid_skills
        return data
    
    experience_min_years: Optional[int] = Field(
        None, description="Ogólne min. lata doświadczenia. '5+' → 5."
    )
    education: Optional[Education] = Field(
        None, description="None jeśli nie wspomniane."
    )

    # ── WYNAGRODZENIE (3 pola) ───────────────────────────
    salary: Optional[Salary] = Field(
        None, description="None jeśli nie podane liczbowo."
    )
    offers_equity: bool = Field(
        False, description="RSU, ESOP, stock options."
    )
    benefits: List[Benefit] = Field(
        default_factory=list,
        description="Znormalizowane benefity wymienione w ogłoszeniu.",
    )

    # ── FIRMA (2 pola) ───────────────────────────────────
    company_name: Optional[str] = Field(None)
    industry: Optional[Industry] = Field(
        None, description="Wnioskuj z opisu firmy/produktu."
    )

## Sprawdzamy?

In [ ]:
sample = df[ df.index == 7 ] #możesz wybierać/losować/srpawdzać inne
sample_content = sample["raw_description"].values[0]

## Bielik

In [ ]:
result = call_api(
    [{"role": "user", "content": sample_title}], 
    JobPosting, 
    model="speakleash/Bielik-11B-v3.0-Instruct"
)


#print(result.model_dump_json(indent=2))
display(JSON(result.model_dump()))

## Co mamy?

Zwroć uwagę, że może to drzewko rozwinac i zagdal w różne gałęzi.

Zobacz, to co mieliśĶy na wejściu -> to ściana z tekstem, na wyjściu mamy dość rozbudowana strukturę.


Zrozum mnie dobrze, tu nie chdozi o to, aby zawsze na siłe budować żlożóne sturktury, ale zwykel tak jest, jak robisz coś trudniejszego, to tak strutkrua rozrasta się dość szybko. Czasem jak struktura robi się zbyg skomplikowana, to można dzielić je na mniejsz kawałki i potem scalać. 

Co do scalania, to jest też ciekwa rzecz, bo jak masz dwie sturktury, to łatwiej to robić. Np. na poziomie 4 pojaiwa się pamięc, i tam trzeba jest sturkutra, ale aby nie gubić się w tym,zwykle ta struktura mocno koroeluje z cześć tego co jest w strcuture output, wtedy możńa łatwo laczyć trudne rzeczy (koncept).


### Podsumowanie

W ramach DWthon udało się przestawić koncepcje Softare 3.0. Każdy numerek, czyli `1 -> 2 -> 3`, to za każdym razem zmiana paradagmatu. Innymi problemy, które możemy rozwiazac przy pomocy Softawr 3.0 sa nie rozwiazalne na dla klasycznego IT. Jak robi się zby tproste rzeczy na poziomie 2 to czasem wydaje się, że da się to rozwiazac kalsycznym IT, ale nawet to co robiśmy w tej lekcji, jak jest ściana tekstu, nie da się tego ogarnac w tak stabilny regexem lub jakiśm ifem. Rożnorodność jest za duża. Na kolejnych poziomach pojawia się dodatkowe możliwości, jescze bardziej wyrózniaja na tle klasycznego IT.


W Software 3.0 rozrożniamy poziomy autuonomii. Mamy ich 7. Wszystko co powyżej 1ki, czyli 2, 3, 4 i 5 moga być production ready. 1k nie może być, bo to sa zapytana ad-hoc (tam musi być czlówiek, który to czyta, to nie jest system, któray działa).


W ramach DWthon zaczeliśmy poziom 2, ale wiele rzecze jeszcze nie udało się pokazać. Sa fajne trick,na które warto zwracać uwage. Np. jak model może myśleć i jak tym sterować. To wsyzstko już jest w dlaszych naszych progrmach edukacyjnych:
- [Praktyczny LLM](https://dataworkshop.eu/pl/praktyczny-llm?utm_source=dwthon&utm_medium=lesson&utm_campaign=llm&utm_term=day3&utm_content=jupyter-notebook) poziom 2 i 3 (5 tygodni) - ruszamy 16 lutego
- Agentic AI: poziom 4 i 5 (5 tygodni)


To jest program 100% online.

<img src="../images/7_levels.png" width="1000px" />


## Bielik na sterydach (Partner technologiczny)

Pod maską DWthon dzieje się sporo. Spodziewaliśmy się dużego ruchu, więc nasza infrastruktura oparta na Kubernetesie musiała być gotowa na wszystko. Ale same kontenery to nie wszystko – potrzebna jest **moc obliczeniowa**.

W naszym Labie działa to tak, że mamy różnych dostawców. Kiedy w kodzie zmieniasz `gemini`, `gpt-4.1` na `Bielik`, Twoje zapytanie fizycznie leci w inne miejsce.
W przypadku **Bielika**, całą obsługę (czyli te tysiące zapytań, które generujecie) wzięła na siebie firma **Beyond.pl**.

Dlaczego o tym wspominam? Bo dostęp do wydajnych klastrów GPU w Polsce to nadal rzadkość. Karty graficzne tej klasy są drogie i trudno dostępne, a Beyond.pl to jedno z niewielu miejsc w kraju, które oferuje taką usługę na światowym poziomie.

Wielkie dzięki dla ekipy z Beyond za wsparcie polskiego open-source i udostępnienie infrastruktury pod nasze eksperymenty! 

## Co dalej? 


Piątek trzynastego i "Fail & fix"

Mamy już trzeci dzień naszego wyzwania. Ale skoro w kalendarzu wypada piątek trzynastego, to idealna okazja, żeby porozmawiać o tym, co w projektach AI lubi się... posypać.

Widzę na Slacku masę świetnych pytań. Postanowiliśmy zrobić z zespołem szybką akcję. Organizujemy dodatkowe spotkanie, ale nie będzie to typowe "podsumowanie".

### Nazwaliśmy to **Fail & Fix**.

Dlaczego? Bo w Software 3.0 nie chodzi o to, by unikać błędów za wszelką cenę. LLM-y z natury są niedoskonałe i – mówiąc wprost – będą popełniać błędy. Cała sztuka polega na tym, by te słabe punkty znać i zaprojektować system tak, by mimo to dowoził wartość.

### Czym jest Fail?

To moment zderzenia z rzeczywistością. Często wynika z:
1. błędnych założeń na starcie,
2. braku metryk (albo mierzenia niewłaściwych rzeczy),
3. niedoszacowania niuansów, które okazują się kluczowe,
4. niezrozumienia, czym tak naprawdę różni się Software 3.0 od klasycznego kodu.

### Czym jest Fix?
To nie jest łatanie dziur taśmą klejącą. To świadoma decyzja architektoniczna, procesowa lub biznesowa. To właśnie te decyzje przesuwają projekt z fazy "fajne demo" do fazy "system, który na siebie zarabia".

Chcemy pokazać historie z naszych projektów – bez lukru i retuszu. Pokażemy, gdzie można się potknąć i jak z tego wybrnąć. Wiedza przyda się zarówno osobom technicznym, jak i tym, które zarządzają projektami czy biznesem. Cel jest jeden: wyjść z notebooka na produkcję.

**[Widzimy się w piątek](https://www.youtube.com/live/VFzWuGcbHxc), 13-tego, o godzinie 9:00.**


In [ ]:
%%html

<iframe style="height:700px;width:100%" src="https://www.youtube.com/embed/VFzWuGcbHxc" frameborder="0" allow="autoplay; encrypted-media" allowfullscreen></iframe>